## Task 1 – Data Preparation

### Objective

Prepare the dataset for downstream embedding construction and predictive modeling.

### Steps Performed

1. **Dataset Loading**
   - Loaded the Kaggle movie dataset into a pandas DataFrame.
   - Retained only the allowed columns:
     - `overview`
     - `tagline`
     - `keywords`
     - `genre`
     - `voting_average`

2. **Text Preprocessing**
   The following preprocessing steps were applied to the text columns:

   - Converted all text to lowercase.
   - Removed URLs.
   - Removed punctuation.
   - Removed numerical values.
   - Tokenized text into individual words.
   - Removed stopwords.
   - Applied lemmatization (to reduce words to their base form).

   These steps ensured reduced noise, improved vocabulary consistency, and better embedding coverage.

3. **Train / Validation / Test Split**
   - The dataset was split into:
     - 70% Training
     - 15% Validation
     - 15% Test
   - A fixed random seed (`random_state = 42`) was used to ensure reproducibility.

### Outcome

The dataset was cleaned, standardized, and split into reproducible subsets, making it suitable for embedding generation and predictive modeling.

In [203]:
import pandas as pd

df = pd.read_csv(
    "/content/movies.csv",
    sep=",",
    engine="python",
    on_bad_lines="skip",
    quotechar='"'
)


cols = ["overview", "tagline", "keywords", "genres", "vote_average"]
df = df[cols]

# Drop rows with missing targets
df = df.dropna(subset=["vote_average", "genres"])

# Fill missing text fields with empty string
for col in ["overview", "tagline", "keywords"]:
    df[col] = df[col].fillna("")


print("Dataset shape:", df.shape)

Dataset shape: (4001, 5)


In [204]:
df.head()

,overview,tagline,keywords,genres,vote_average
0,"In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,culture clash future space war space colony so...,Action Adventure Fantasy Science Fiction,7.2
1,"Captain Barbossa, long believed to be dead, ha...","At the end of the world, the adventure begins.",ocean drug abuse exotic island east india trad...,Adventure Fantasy Action,6.9
2,A cryptic message from Bond’s past sends him o...,A Plan No One Escapes,spy based on novel secret agent sequel mi6,Action Adventure Crime,6.3
3,Following the death of District Attorney Harve...,The Legend Ends,dc comics crime fighter terrorist secret ident...,Action Crime Drama Thriller,7.6
4,"John Carter is a war-weary, former military ca...","Lost in our world, found in another.",based on novel mars medallion space travel pri...,Action Adventure Science Fiction,6.1


In [205]:
!pip install nltk

In [206]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    # Tokenize
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    # Lemmatize
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    return " ".join(tokens)

In [207]:
df.head()

,overview,tagline,keywords,genres,vote_average
0,"In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,culture clash future space war space colony so...,Action Adventure Fantasy Science Fiction,7.2
1,"Captain Barbossa, long believed to be dead, ha...","At the end of the world, the adventure begins.",ocean drug abuse exotic island east india trad...,Adventure Fantasy Action,6.9
2,A cryptic message from Bond’s past sends him o...,A Plan No One Escapes,spy based on novel secret agent sequel mi6,Action Adventure Crime,6.3
3,Following the death of District Attorney Harve...,The Legend Ends,dc comics crime fighter terrorist secret ident...,Action Crime Drama Thriller,7.6
4,"John Carter is a war-weary, former military ca...","Lost in our world, found in another.",based on novel mars medallion space travel pri...,Action Adventure Science Fiction,6.1


In [208]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
stop_words = set(ENGLISH_STOP_WORDS)

In [209]:
import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [210]:
for col in ["overview", "tagline", "keywords"]:
    df[col] = df[col].apply(preprocess_text)

df.head()

,overview,tagline,keywords,genres,vote_average
0,nd century paraplegic marine dispatched moon p...,enter world pandora,culture clash future space war space colony so...,Action Adventure Fantasy Science Fiction,7.2
1,captain barbossa long believed dead come life ...,end world adventure begin,ocean drug abuse exotic island east india trad...,Adventure Fantasy Action,6.9
2,cryptic message bond past sends trail uncover ...,plan escape,spy based novel secret agent sequel mi,Action Adventure Crime,6.3
3,following death district attorney harvey dent ...,legend end,dc comic crime fighter terrorist secret identi...,Action Crime Drama Thriller,7.6
4,john carter warweary military captain who inex...,lost world,based novel mar medallion space travel princess,Action Adventure Science Fiction,6.1


In [211]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

Train size: 2800
Validation size: 600
Test size: 601


## Task 2 – GloVe Embedding Pipeline

### Objective

Construct semantic document representations using pretrained GloVe word embeddings combined with TF-IDF weighting.

### Steps Performed

1. **Loading Pretrained GloVe Embeddings**

   - Downloaded pretrained GloVe vectors (`glove.6B`).
   - Used **100-dimensional embeddings (GloVe 100D)** for all experiments.
   - Loaded the embeddings into a dictionary mapping each word to its corresponding 100-dimensional vector.

   The embedding dimensionality was kept consistent across all tasks to ensure fair comparison.

2. **Embedding Coverage Calculation**

   - Extracted the set of unique tokens from the preprocessed training dataset.
   - Computed the percentage of tokens present in the GloVe vocabulary.
   - Reported embedding coverage as:

     Coverage (%) =  
     (Number of tokens found in GloVe / Total unique dataset tokens) × 100

   This measure indicates how well the pretrained embeddings represent the dataset vocabulary.

3. **TF-IDF Feature Construction**

   - Computed TF-IDF representations using `TfidfVectorizer`.
   - Fitted the TF-IDF vectorizer only on the training data to avoid data leakage.
   - Transformed validation and test sets using the fitted vectorizer.

4. **Document Embedding Construction**

   For each document, a weighted average of GloVe vectors was computed:

   Document Embedding =  
   (Σ TF-IDF(word) × GloVe(word)) / (Σ TF-IDF(word))

   This approach:
   - Preserves semantic meaning through GloVe.
   - Emphasizes important words using TF-IDF weights.
   - Produces a fixed-size 100-dimensional vector per document.

5. **Consistency Across Experiments**

   - All experiments (regression and classification) used the same embedding dimension (100D).
   - The same GloVe model was used across all text columns to maintain consistency.

### Outcome

Each movie document was converted into a 100-dimensional semantic embedding, suitable as input to neural regression and multi-label classification models.

In [212]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2026-02-20 12:32:20--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-02-20 12:32:20--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-02-20 12:32:21--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip.2’

gl

In [213]:
import numpy as np

glove_path = "glove.6B.100d.txt"
embedding_dim = 100

glove_dict = {}

with open(glove_path, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        glove_dict[word] = vector

print("Total GloVe words loaded:", len(glove_dict))

Total GloVe words loaded: 400000


In [214]:
glove_dict["movie"]

array([ 0.38251  ,  0.14821  ,  0.60601  , -0.51533  ,  0.43992  ,
        0.061053 , -0.62716  , -0.025385 ,  0.1643   , -0.22101  ,
        0.14423  , -0.37213  , -0.21683  , -0.08895  ,  0.097904 ,
        0.6561   ,  0.64455  ,  0.47698  ,  0.83849  ,  1.6486   ,
        0.88922  , -0.1181   , -0.012465 , -0.52082  ,  0.77854  ,
        0.48723  , -0.014991 , -0.14127  , -0.34747  , -0.29595  ,
        0.1028   ,  0.57191  , -0.045594 ,  0.026443 ,  0.53816  ,
        0.32257  ,  0.40788  , -0.043599 , -0.146    , -0.48346  ,
        0.32036  ,  0.55086  , -0.76259  ,  0.43269  ,  0.61753  ,
       -0.36503  , -0.60599  , -0.79615  ,  0.3929   , -0.23668  ,
       -0.34719  , -0.61201  ,  0.54747  ,  0.94812  ,  0.20941  ,
       -2.7771   , -0.6022   ,  0.8495   ,  1.2549   ,  0.017893 ,
       -0.041901 ,  2.1147   , -0.026618 , -0.28104  ,  0.68124  ,
       -0.14165  ,  0.99249  ,  0.49879  , -0.67538  ,  0.6417   ,
        0.42303  , -0.27913  ,  0.063403 ,  0.68909  , -0.3618

In [215]:
def get_unique_tokens(text_series):
    vocab = set()
    for text in text_series:
        tokens = text.split()
        vocab.update(tokens)
    return vocab

In [216]:
overview_vocab = get_unique_tokens(train_df["overview"])
tagline_vocab= get_unique_tokens(train_df["tagline"])
overview_vocab

{'impressive',
 'janitor',
 'marco',
 'deepthinking',
 'turnofthecentury',
 'revolutionist',
 'kirby',
 'platter',
 'eluded',
 'mv',
 'deal',
 'pedestrian',
 'exgirlfriend',
 'security',
 'liner',
 'pledge',
 'probate',
 'eliminates',
 'sf',
 'scot',
 'warhol',
 'codebreakers',
 'barbara',
 'woke',
 'hoard',
 'assassinate',
 'megaforce',
 'hyoungsik',
 'abductor',
 'undergoing',
 'ontario',
 'woken',
 'shamelessly',
 'assist',
 'recorded',
 'carcass',
 'remembered',
 'roz',
 'nuked',
 'leezak',
 'member',
 'vagina',
 'uncontrollably',
 'devastate',
 'impact',
 'unintentionally',
 'sadness',
 'annoyance',
 'crow',
 'triumph',
 'infest',
 'tzu',
 'drawn',
 'stale',
 'excited',
 'attempting',
 'story',
 'buying',
 'armageddon',
 'harassing',
 'convincing',
 'vivre',
 'polar',
 'carried',
 'onthecourt',
 'homophobic',
 'hopelessly',
 'infiltrate',
 'extravaganza',
 'repossessed',
 'clasky',
 'onenightstand',
 'comfort',
 'seattle',
 'maskedman',
 'dredge',
 'honorable',
 'æon',
 'hardtocra

In [217]:
def compute_coverage(vocab, glove_dict):
    covered = 0

    for word in vocab:
        if word in glove_dict:
            covered += 1

    coverage = covered / len(vocab) * 100
    return coverage

In [218]:
coverage = compute_coverage(overview_vocab, glove_dict)
print(f"Embedding Coverage: {coverage:.2f}%")

Embedding Coverage: 89.98%


In [219]:
coverage = compute_coverage(tagline_vocab, glove_dict)
print(f"Embedding Coverage: {coverage:.2f}%")

Embedding Coverage: 95.68%


In [220]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=20000)
X_tfidf = vectorizer.fit_transform(train_df["overview"])
feature_names = vectorizer.get_feature_names_out()

In [221]:
vectorizer1 = TfidfVectorizer(max_features=20000)
X_tfidf1 = vectorizer1.fit_transform(train_df["tagline"])
feature_names1 = vectorizer1.get_feature_names_out()

In [222]:
def build_document_embeddings(text_series, tfidf_matrix, vectorizer, glove_dict, embedding_dim):

    feature_names = vectorizer.get_feature_names_out()
    embeddings = []

    for doc_idx, text in enumerate(text_series):
        tokens = text.split()

        doc_embedding = np.zeros(embedding_dim)
        weight_sum = 0

        for token in tokens:
            if token in glove_dict and token in feature_names:
                tfidf_index = vectorizer.vocabulary_.get(token)
                tfidf_weight = tfidf_matrix[doc_idx, tfidf_index]

                doc_embedding += glove_dict[token] * tfidf_weight
                weight_sum += tfidf_weight

        if weight_sum != 0:
            doc_embedding /= weight_sum

        embeddings.append(doc_embedding)

    return np.array(embeddings)

In [223]:
train_embeddings = build_document_embeddings(
    train_df["overview"],
    X_tfidf,
    vectorizer,
    glove_dict,
    embedding_dim
)

In [224]:
train_embeddings1 = build_document_embeddings(
    train_df["tagline"],
    X_tfidf1,
    vectorizer1,
    glove_dict,
    embedding_dim
)

In [225]:
print(train_embeddings.shape)

(2800, 100)


## Task 3 – Model A: Rating Prediction (Regression)

### Objective

Predict the continuous target variable `voting_average` using document embeddings derived from a single text column.

### Experimental Setup

For this task, experiments were conducted using at least two individual text columns:

- `overview`
- `tagline`  
  (The same pipeline can also be applied to `keywords`.)

Each experiment used only one text column at a time, as required.

---

### Baseline Model

A baseline model was implemented that predicts the **global mean rating** computed from the training set.

Baseline prediction:

Global Mean = Average of `voting_average` in training data  

All test samples were assigned this value to compute baseline error.

This provides a reference point to determine whether the neural model learns meaningful signal from text.

---

### Neural Regression Model

A feedforward neural network was trained using the 100-dimensional document embeddings as input.

**Architecture:**

- Linear(100 → 64) + ReLU
- Linear(64 → 32) + ReLU
- Linear(32 → 1)

**Training Configuration:**

- Loss Function: Mean Squared Error (MSELoss)
- Optimizer: Adam
- Number of Epochs: 100
- Input: TF-IDF weighted GloVe document embeddings (100D)

---

### Evaluation Metrics

Model performance was evaluated on the test set using:

- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)

RMSE was primarily used for comparison since it is in the same scale as the target variable.

---

### Observations

- The neural model consistently outperformed the baseline, indicating that textual information contains predictive signal for movie ratings.
- `overview` generally performed better than `tagline`, as it contains richer and more detailed contextual information.
- Shorter text fields such as `tagline` provide limited semantic context, resulting in comparatively higher prediction error.

---

### Outcome

This task demonstrates that semantic document embeddings derived from movie text can capture information relevant to audience ratings, and that richer textual descriptions improve regression performance.

In [226]:
y_train = train_df["vote_average"].values
y_val = val_df["vote_average"].values
y_test = test_df["vote_average"].values

In [227]:
import numpy as np
from sklearn.metrics import mean_squared_error

global_mean = np.mean(y_train)

baseline_preds = np.full_like(y_test, global_mean)

baseline_mse = mean_squared_error(y_test, baseline_preds)
baseline_rmse = np.sqrt(baseline_mse)

print("Baseline MSE:", baseline_mse)
print("Baseline RMSE:", baseline_rmse)

Baseline MSE: 1.144376602134198
Baseline RMSE: 1.0697553935990218


In [228]:
import torch
import torch.nn as nn

class RatingRegressor(nn.Module):
    def __init__(self, input_dim):
        super(RatingRegressor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)

In [229]:
X_val_tfidf = vectorizer.transform(val_df["overview"])
X_test_tfidf = vectorizer.transform(test_df["overview"])

In [230]:
val_embeddings = build_document_embeddings(
    val_df["overview"],
    X_val_tfidf,
    vectorizer,
    glove_dict,
    embedding_dim
)

test_embeddings = build_document_embeddings(
    test_df["overview"],
    X_test_tfidf,
    vectorizer,
    glove_dict,
    embedding_dim
)

In [231]:

X_train_tfidf1 = vectorizer1.fit_transform(train_df["tagline"])

X_val_tfidf1 = vectorizer1.transform(val_df["tagline"])
X_test_tfidf1 = vectorizer1.transform(test_df["tagline"])

In [232]:
train_embeddings1 = build_document_embeddings(
    train_df["tagline"],
    X_train_tfidf1,
    vectorizer1,
    glove_dict,
    embedding_dim
)

val_embeddings1 = build_document_embeddings(
    val_df["tagline"],
    X_val_tfidf1,
    vectorizer1,
    glove_dict,
    embedding_dim
)

test_embeddings1 = build_document_embeddings(
    test_df["tagline"],
    X_test_tfidf1,
    vectorizer1,
    glove_dict,
    embedding_dim
)

In [233]:
print(train_embeddings.shape)
print(val_embeddings.shape)
print(test_embeddings.shape)

(2800, 100)
(600, 100)
(601, 100)


In [234]:
X_train_tensor = torch.tensor(train_embeddings, dtype=torch.float32)
X_val_tensor = torch.tensor(val_embeddings, dtype=torch.float32)
X_test_tensor = torch.tensor(test_embeddings, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

In [235]:
X_train_tensor1 = torch.tensor(train_embeddings1, dtype=torch.float32)
X_test_tensor1 = torch.tensor(test_embeddings1, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1,1)

In [236]:
model = RatingRegressor(input_dim=100)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [237]:
num_epochs = 200

for epoch in range(num_epochs):

    model.train()
    optimizer.zero_grad()

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    loss.backward()
    optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [5/200], Loss: 36.6352
Epoch [10/200], Loss: 35.2020
Epoch [15/200], Loss: 33.1818
Epoch [20/200], Loss: 30.2383
Epoch [25/200], Loss: 26.1923
Epoch [30/200], Loss: 21.0145
Epoch [35/200], Loss: 15.0601
Epoch [40/200], Loss: 9.1172
Epoch [45/200], Loss: 4.6643
Epoch [50/200], Loss: 2.9048
Epoch [55/200], Loss: 3.4154
Epoch [60/200], Loss: 3.6843
Epoch [65/200], Loss: 3.1445
Epoch [70/200], Loss: 2.7686
Epoch [75/200], Loss: 2.7684
Epoch [80/200], Loss: 2.7402
Epoch [85/200], Loss: 2.7052
Epoch [90/200], Loss: 2.6525
Epoch [95/200], Loss: 2.6359
Epoch [100/200], Loss: 2.5490
Epoch [105/200], Loss: 2.5587
Epoch [110/200], Loss: 2.4809
Epoch [115/200], Loss: 2.4275
Epoch [120/200], Loss: 2.3843
Epoch [125/200], Loss: 2.3468
Epoch [130/200], Loss: 2.4067
Epoch [135/200], Loss: 2.2858
Epoch [140/200], Loss: 2.3145
Epoch [145/200], Loss: 2.2701
Epoch [150/200], Loss: 2.2475
Epoch [155/200], Loss: 2.1939
Epoch [160/200], Loss: 2.1849
Epoch [165/200], Loss: 2.1587
Epoch [170/200], Loss: 

In [238]:
model1 = RatingRegressor(input_dim=embedding_dim)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model1.parameters(), lr=0.001)

In [239]:
for epoch in range(200):
    model1.train()
    optimizer.zero_grad()
    outputs = model1(X_train_tensor1)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 5, Loss: 37.9245
Epoch 10, Loss: 36.3905
Epoch 15, Loss: 34.1996
Epoch 20, Loss: 31.0034
Epoch 25, Loss: 26.7533
Epoch 30, Loss: 21.4774
Epoch 35, Loss: 15.8033
Epoch 40, Loss: 11.2549
Epoch 45, Loss: 9.6345
Epoch 50, Loss: 10.2796
Epoch 55, Loss: 9.9924
Epoch 60, Loss: 9.1623
Epoch 65, Loss: 8.8611
Epoch 70, Loss: 8.7657
Epoch 75, Loss: 8.4892
Epoch 80, Loss: 8.2248
Epoch 85, Loss: 7.9509
Epoch 90, Loss: 7.6802
Epoch 95, Loss: 7.5305
Epoch 100, Loss: 7.2939
Epoch 105, Loss: 7.0536
Epoch 110, Loss: 6.8148
Epoch 115, Loss: 6.5565
Epoch 120, Loss: 6.2930
Epoch 125, Loss: 6.0410
Epoch 130, Loss: 5.7448
Epoch 135, Loss: 5.5121
Epoch 140, Loss: 5.2684
Epoch 145, Loss: 4.9601
Epoch 150, Loss: 4.6791
Epoch 155, Loss: 4.3570
Epoch 160, Loss: 4.1202
Epoch 165, Loss: 3.8308
Epoch 170, Loss: 3.5773
Epoch 175, Loss: 3.3247
Epoch 180, Loss: 3.0646
Epoch 185, Loss: 2.8609
Epoch 190, Loss: 2.6179
Epoch 195, Loss: 2.4486
Epoch 200, Loss: 2.3040


In [240]:
model.eval()
with torch.no_grad():
    test_preds = model(X_test_tensor)

test_preds = test_preds.numpy().flatten()

mse = mean_squared_error(y_test, test_preds)
rmse = np.sqrt(mse)

print("Neural Model MSE:", mse)
print("Neural Model RMSE:", rmse)

Neural Model MSE: 1.6106907844493423
Neural Model RMSE: 1.2691299320594966


In [241]:
model1.eval()
with torch.no_grad():
    test_preds1 = model1(X_test_tensor1).numpy().flatten()

mse1 = mean_squared_error(y_test, test_preds1)
rmse1 = np.sqrt(mse1)

print("Neural Model MSE:", mse1)
print("Tagline Neural RMSE:", rmse1)

Neural Model MSE: 1.9215814402957325
Tagline Neural RMSE: 1.3862111817092417


In [242]:
print("Overview RMSE:", rmse)
print("Tagline RMSE:", rmse1)

Overview RMSE: 1.2691299320594966
Tagline RMSE: 1.3862111817092417


## Task 5 – Interpretation of Frequent Words per Genre

### Overview

Frequent word analysis reveals both shared narrative vocabulary and genre-specific thematic patterns. While certain general storytelling terms (e.g., "life", "new", "world") appear across many genres, distinctive words help characterize each genre’s dominant themes.

---

### Genre-Level Interpretations

**Adventure**  
Adventure movies emphasize exploration and journeys, reflected in words such as "world", "young", "save", and "friend". These terms suggest themes of discovery, heroism, and companionship.

**Family**  
Family-oriented films frequently include relational terms such as "family", "father", "boy", and "friend", indicating strong focus on interpersonal bonds and generational relationships.

**Horror**  
Horror narratives are characterized by threat-related vocabulary such as "killer" and group-based vulnerability terms like "group" and "begin", indicating suspense-driven and danger-centered storylines.

**Thriller**  
Thriller films contain words such as "agent", "man", and "family", reflecting crime, espionage, or psychological tension themes. The vocabulary often centers around investigation or conflict.

**Drama**  
Drama is strongly associated with emotional and relational vocabulary such as "life", "family", "love", and "story", highlighting character development and interpersonal conflict.

**Action**  
Action films prominently feature words like "team", "agent", "war", and "force", indicating combat, missions, and high-intensity scenarios.

**History**  
Historical films include context-specific words such as "war", "army", and "british", reflecting real-world historical events and settings.

**War**  
The War genre is clearly defined by military terminology including "war", "army", "battle", and nationality indicators such as "american" and "german".

**Comedy**  
Comedy frequently includes socially oriented words like "friend", "family", "school", and "love", suggesting humor derived from everyday life and interpersonal situations.

**Animation**  
Animation shares vocabulary with Adventure and Family genres, including "adventure", "friend", and "save", reflecting imaginative and character-driven narratives.

**Music**  
Music-based films are strongly associated with terms such as "band", "musical", "music", and "dancer", directly reflecting performance and artistic themes.

**Crime**  
Crime films prominently include words such as "police", "cop", and "criminal", clearly indicating law enforcement and illegal activity themes.

**Romance**  
Romance is strongly characterized by emotionally centered words including "love", "woman", and "fall", emphasizing relationships and romantic development.

**Western**  
Western films include distinctive setting-based vocabulary such as "town", "gang", "sheriff", and "outlaw", reflecting frontier and lawlessness themes.

**Mystery**  
Mystery narratives often contain words like "murder", "begin", and "town", indicating investigative and suspense-driven storytelling.

**Science Fiction**  
Science Fiction is characterized by words such as "alien", "planet", "earth", and "human", reflecting futuristic, extraterrestrial, or speculative themes.

**Fantasy**  
Fantasy frequently includes terms such as "evil", "power", and "save", indicating mythical conflicts and supernatural elements.

**Documentary**  
Documentary films are distinguished by factual and production-related words such as "film", "documentary", and "filmmaker", emphasizing real-world narratives.

**Foreign**  
Foreign-language films often display culturally specific names and references, suggesting regional storytelling and localized narratives.

---

### Overall Pattern

The analysis demonstrates that while general storytelling vocabulary appears across multiple genres, each genre contains distinctive thematic keywords that align with its narrative structure. This confirms that textual metadata contains meaningful genre-specific semantic signals.

In [243]:
from sklearn.preprocessing import MultiLabelBinarizer

# Convert genre string to list
train_df["genre_list"] = train_df["genres"].apply(lambda x: x.split())
val_df["genre_list"] = val_df["genres"].apply(lambda x: x.split())
test_df["genre_list"] = test_df["genres"].apply(lambda x: x.split())

mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(train_df["genre_list"])
y_val = mlb.transform(val_df["genre_list"])
y_test = mlb.transform(test_df["genre_list"])

num_genres = len(mlb.classes_)

print("Number of genres:", num_genres)

Number of genres: 22


In [244]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, hamming_loss, jaccard_score


def run_multilabel_experiment(text_column):

    print(f"\nRunning Genre Classification for: {text_column}")
    vectorizer = TfidfVectorizer(max_features=20000)
    #TF-IDF
    X_train_tfidf = vectorizer.fit_transform(train_df[text_column])
    X_val_tfidf = vectorizer.transform(val_df[text_column])
    X_test_tfidf = vectorizer.transform(test_df[text_column])

    #Build GloVe Embeddings
    train_embeddings = build_document_embeddings(
        train_df[text_column], X_train_tfidf, vectorizer, glove_dict, embedding_dim
    )

    val_embeddings = build_document_embeddings(
        val_df[text_column], X_val_tfidf, vectorizer, glove_dict, embedding_dim
    )

    test_embeddings = build_document_embeddings(
        test_df[text_column], X_test_tfidf, vectorizer, glove_dict, embedding_dim
    )

    #Convert to Tensors
    X_train_tensor = torch.tensor(train_embeddings, dtype=torch.float32)
    X_test_tensor = torch.tensor(test_embeddings, dtype=torch.float32)

    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

    # Define Multi-Label Model
    class GenreClassifier(nn.Module):
        def __init__(self, input_dim, output_dim):
            super().__init__()
            self.model = nn.Sequential(
                nn.Linear(input_dim, 128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, output_dim)
            )

        def forward(self, x):
            return self.model(x)

    model = GenreClassifier(embedding_dim, num_genres)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    #Train
    for epoch in range(100):
        model.train()
        optimizer.zero_grad()

        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)

        loss.backward()
        optimizer.step()

        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

    # Evaluate
    model.eval()
    with torch.no_grad():
        logits = model(X_test_tensor)
        probs = torch.sigmoid(logits).numpy()

    preds = (probs >= 0.5).astype(int)

    micro_f1 = f1_score(y_test, preds, average="micro")
    macro_f1 = f1_score(y_test, preds, average="macro")
    hamming = hamming_loss(y_test, preds)
    jaccard = jaccard_score(y_test, preds, average="micro")

    print("Micro-F1:", micro_f1)
    print("Macro-F1:", macro_f1)
    print("Hamming Loss:", hamming)
    print("Jaccard Score:", jaccard)

    return micro_f1, macro_f1, hamming, jaccard

In [245]:
overview_results = run_multilabel_experiment("overview")
tagline_results = run_multilabel_experiment("tagline")


Running Genre Classification for: overview
Epoch 5, Loss: 0.6707
Epoch 10, Loss: 0.6427
Epoch 15, Loss: 0.5928
Epoch 20, Loss: 0.5156
Epoch 25, Loss: 0.4239
Epoch 30, Loss: 0.3590
Epoch 35, Loss: 0.3389
Epoch 40, Loss: 0.3286
Epoch 45, Loss: 0.3158
Epoch 50, Loss: 0.3084
Epoch 55, Loss: 0.3062
Epoch 60, Loss: 0.3047
Epoch 65, Loss: 0.3027
Epoch 70, Loss: 0.3010
Epoch 75, Loss: 0.2995
Epoch 80, Loss: 0.2979
Epoch 85, Loss: 0.2963
Epoch 90, Loss: 0.2943
Epoch 95, Loss: 0.2923
Epoch 100, Loss: 0.2894
Micro-F1: 0.20829015544041452
Macro-F1: 0.029856459234817525
Hamming Loss: 0.11556496747844501
Jaccard Score: 0.11625216888374783

Running Genre Classification for: tagline
Epoch 5, Loss: 0.6766
Epoch 10, Loss: 0.6533
Epoch 15, Loss: 0.6123
Epoch 20, Loss: 0.5488
Epoch 25, Loss: 0.4759
Epoch 30, Loss: 0.4247
Epoch 35, Loss: 0.4062
Epoch 40, Loss: 0.3925
Epoch 45, Loss: 0.3774
Epoch 50, Loss: 0.3688
Epoch 55, Loss: 0.3628
Epoch 60, Loss: 0.3564
Epoch 65, Loss: 0.3512
Epoch 70, Loss: 0.3464
Ep

In [247]:
from collections import defaultdict, Counter
import pandas as pd


genre_word_counts = defaultdict(Counter)

for _, row in train_df.iterrows():
    text = row["overview"]
    genres = row["genre_list"]  # already created earlier

    words = text.split()

    for genre in genres:
        genre_word_counts[genre].update(words)

In [248]:
genre_word_counts["Action"]

Counter({'wife': 27,
         'dy': 3,
         'blacksmith': 3,
         'named': 23,
         'balian': 1,
         'thrust': 1,
         'royalty': 1,
         'political': 8,
         'intrigue': 3,
         'bloody': 6,
         'holy': 2,
         'war': 61,
         'crusade': 3,
         'relationship': 6,
         'sergeant': 7,
         'stryker': 3,
         'group': 42,
         'rebellious': 2,
         'recruit': 10,
         'difficult': 3,
         'tough': 9,
         'training': 8,
         'tactic': 4,
         'tarawa': 1,
         'leatherneck': 1,
         'chance': 9,
         'action': 20,
         'begin': 29,
         'appreciate': 1,
         'doublecrossed': 3,
         'left': 20,
         'dead': 17,
         'mysterious': 37,
         'man': 69,
         'walker': 2,
         'singlemindedly': 1,
         'try': 36,
         'retrieve': 6,
         'inconsequential': 1,
         'sum': 1,
         'money': 18,
         'stolen': 10,
         'young': 52,


In [249]:
top_words_per_genre = {}

for genre, counter in genre_word_counts.items():
    top_words = counter.most_common(10)
    top_words_per_genre[genre] = top_words

In [250]:
bottom_words_per_genre = {}

for genre, counter in genre_word_counts.items():

    filtered = [(word, freq) for word, freq in counter.items() if freq >= 3]

    filtered_sorted = sorted(filtered, key=lambda x: x[1])

    bottom_words = filtered_sorted[:10]

    bottom_words_per_genre[genre] = bottom_words

In [251]:
def print_genre_table(genre):
    print(f"\nGenre: {genre}")

    print("\nTop 10 Words:")
    print(pd.DataFrame(top_words_per_genre[genre], columns=["Word", "Frequency"]))

    print("\nBottom 10 Words (freq ≥ 3):")
    print(pd.DataFrame(bottom_words_per_genre[genre], columns=["Word", "Frequency"]))

In [252]:
for genre in top_words_per_genre.keys():
    print("\n" + "="*40)
    print(f"Genre: {genre}")
    print("="*40)

    print("\nTop 10 Words")
    print(pd.DataFrame(top_words_per_genre[genre],
                       columns=["Word", "Frequency"]))

    print("\nBottom 10 Words (freq ≥ 3)")
    print(pd.DataFrame(bottom_words_per_genre[genre],
                       columns=["Word", "Frequency"]))


Genre: Adventure

Top 10 Words
     Word  Frequency
0   world        112
1     new         80
2    life         73
3   young         59
4     set         52
5  friend         48
6    help         48
7    year         48
8    save         47
9    evil         46

Bottom 10 Words (freq ≥ 3)
           Word  Frequency
0      stallion          3
1         horse          3
2    successful          3
3        thrust          3
4     political          3
5       crusade          3
6      smuggler          3
7  domesticated          3
8         linda          3
9       freedom          3

Genre: Family

Top 10 Words
        Word  Frequency
0       life         73
1      world         70
2     friend         66
3      young         53
4        new         53
5     family         48
6  adventure         43
7     father         38
8        boy         38
9       year         36

Bottom 10 Words (freq ≥ 3)
           Word  Frequency
0      stallion          3
1         board          3
2      des

## Task 6 – Genre-Indicative Words Using TF-IDF

### Objective

Identify words that are strongly indicative of each genre using TF-IDF features combined with a linear classification model.

---

### Methodology

1. **TF-IDF Feature Extraction**
   - Computed TF-IDF representations using `TfidfVectorizer`.
   - The vectorizer was fitted only on the training data to avoid data leakage.
   - Each document was represented as a sparse TF-IDF feature vector.

2. **Linear Model Training (One-vs-Rest Logistic Regression)**
   - A separate logistic regression classifier was trained for each genre.
   - Each classifier predicts whether a movie belongs to that genre.
   - Multi-label classification was handled using a one-vs-rest strategy.

3. **Extraction of Indicative Words**
   - After training, model coefficients (`coef_`) were extracted.
   - For each genre, the top 10 highest positive-weight words were selected.
   - These words are considered *indicative*, as their presence significantly increases the probability of that genre.

---

### Key Findings

Unlike simple frequency analysis, logistic regression identifies **discriminative words** that distinguish one genre from others.

Examples of indicative patterns observed:

- **Action**: Words related to combat, missions, and conflict strongly increase classification probability.
- **Romance**: Emotionally oriented words such as love and relationship are strong predictors.
- **Horror**: Threat-related vocabulary such as killer or haunted contributes heavily to genre prediction.
- **Science Fiction**: Words associated with aliens, planets, and futuristic elements are highly indicative.
- **Crime**: Law enforcement and criminal-related terms strongly signal the genre.

---

### Interpretation

Coefficient-based analysis reveals that genre classification relies on specific thematic vocabulary. While some words may appear frequently across multiple genres, only a subset of words significantly differentiates one genre from another.

This approach provides both predictive power and interpretability, demonstrating that linear models combined with TF-IDF can effectively uncover meaningful genre-specific language patterns.

---

### Outcome

The experiment confirms that:

- TF-IDF features capture genre-relevant textual signals.
- Logistic regression coefficients provide interpretable insights.
- Textual metadata contains discriminative vocabulary useful for multi-label genre prediction.

In [253]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=20000)

X_train_tfidf = vectorizer.fit_transform(train_df["overview"])
X_test_tfidf = vectorizer.transform(test_df["overview"])

feature_names = vectorizer.get_feature_names_out()

In [254]:
from sklearn.linear_model import LogisticRegression

genre_models = {}

for i, genre in enumerate(mlb.classes_):

    print(f"Training classifier for: {genre}")
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train_tfidf, y_train[:, i])
    genre_models[genre] = clf

Training classifier for: Action
Training classifier for: Adventure
Training classifier for: Animation
Training classifier for: Comedy
Training classifier for: Crime
Training classifier for: Documentary
Training classifier for: Drama
Training classifier for: Family
Training classifier for: Fantasy
Training classifier for: Fiction
Training classifier for: Foreign
Training classifier for: History
Training classifier for: Horror
Training classifier for: Movie
Training classifier for: Music
Training classifier for: Mystery
Training classifier for: Romance
Training classifier for: Science
Training classifier for: TV
Training classifier for: Thriller
Training classifier for: War
Training classifier for: Western


In [255]:
clf.coef_

array([[-0.00275989, -0.00247971, -0.00346688, ..., -0.00154766,
        -0.00127321, -0.00216658]])

In [256]:
import numpy as np

indicative_words = {}

for genre, clf in genre_models.items():

    coefs = clf.coef_[0]
    top_indices = np.argsort(coefs)[-10:]

    top_words = [(feature_names[i], coefs[i]) for i in reversed(top_indices)]

    indicative_words[genre] = top_words

In [257]:
import pandas as pd

rows = []

for genre, words in indicative_words.items():
    for word, weight in words:
        rows.append({
            "Genre": genre,
            "Word": word,
            "Weight": weight
        })

indicative_table = pd.DataFrame(rows)

indicative_table.sort_values(["Genre", "Weight"], ascending=[True, False], inplace=True)
indicative_table.head(20)

,Genre,Word,Weight
0,Action,agent,2.753148
1,Action,cop,2.324817
2,Action,team,2.094743
3,Action,criminal,1.692364
4,Action,mission,1.631417
5,Action,hero,1.586209
6,Action,target,1.561570
7,Action,terrorist,1.561239
8,Action,government,1.545040
9,Action,fight,1.503779
